In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load data
df = pd.read_csv("datasets/credit_risk_dataset.csv")  # change to your file name
print("Shape:", df.shape)
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0], "\n")

# 2. Basic cleaning
# Fill missing numeric values with the median
df["person_emp_length"] = df["person_emp_length"].fillna(df["person_emp_length"].median())
df["loan_int_rate"] = df["loan_int_rate"].fillna(df["loan_int_rate"].median())

# Drop unrealistic values (e.g. age 144, employment length 123 years)
df = df[(df["person_age"] <= 100) & (df["person_emp_length"] <= 60)]

# 3. Encode categorical features (one-hot encoding)
categorical = ["person_home_ownership", "loan_intent", "loan_grade", "cb_person_default_on_file"]
df = pd.get_dummies(df, columns=categorical, drop_first=True)

# 4. Split features and target
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Feature scaling (StandardScaler: mean = 0, std = 1)
# Fit ONLY on training data, then apply to both train and test to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Simple scaling example: look at two features before and after
example_cols = ["person_income", "loan_amnt"]
idx = [X.columns.get_loc(c) for c in example_cols]
print("Before scaling (first 5 rows):")
print(X_train[example_cols].head())
print("\nAfter scaling (first 5 rows):")
print(pd.DataFrame(X_train_scaled[:5, idx], columns=example_cols))
print("\nMean after scaling:", X_train_scaled[:, idx].mean(axis=0).round(3))
print("Std after scaling: ", X_train_scaled[:, idx].std(axis=0).round(3), "\n")

# 6. Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# 7. Evaluate
y_pred = model.predict(X_test_scaled)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

# 8. Feature importance (coefficients are comparable because features are scaled)
coefs = pd.Series(model.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print("Top 10 most influential features:\n", coefs.head(10))

Shape: (32581, 12)
Missing values:
 person_emp_length     895
loan_int_rate        3116
dtype: int64 

Before scaling (first 5 rows):
       person_income  loan_amnt
27329         133000      15000
28739          98850      12500
2338           33500       2500
11627          75000       9000
32541         163000      25000

After scaling (first 5 rows):
   person_income  loan_amnt
0       1.304607   0.861277
1       0.641886   0.464512
2      -0.626306  -1.122548
3       0.179050  -0.090959
4       1.886792   2.448336

Mean after scaling: [ 0. -0.]
Std after scaling:  [1. 1.] 

Accuracy: 0.8639

Confusion matrix:
 [[4863  231]
 [ 656  765]]

Classification report:
               precision    recall  f1-score   support

           0       0.88      0.95      0.92      5094
           1       0.77      0.54      0.63      1421

    accuracy                           0.86      6515
   macro avg       0.82      0.75      0.77      6515
weighted avg       0.86      0.86      0.85      6515